# Phase 13.5 — Product / Catalog API

Walk through catalog read provider → serving service → thin GET route.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src" / "productiq").is_dir():
    ROOT = ROOT.parents[1]
sys.path.insert(0, str(ROOT / "src"))

from fastapi.testclient import TestClient

from productiq.api import create_app
from productiq.serving.catalog_read import InMemoryProductCatalogReadProvider
from productiq.serving.product_schema import ProductCatalogReadModel
from productiq.serving.product_service import ProductionProductServingService
from productiq.serving.versioning import SERVING_API_ROUTE_PREFIX

In [ ]:
catalog = InMemoryProductCatalogReadProvider(
    {
        "123456789001": ProductCatalogReadModel(
            product_id="123456789001",
            brand="puma",
            description="Striped Slim Fit Shirt",
            category_gender="Men",
            product_type="shirt",
        )
    }
)
app = create_app(product_service=ProductionProductServingService(catalog))

In [ ]:
with TestClient(app) as client:
    hit = client.get(f"{SERVING_API_ROUTE_PREFIX}/products/123456789001")
    miss = client.get(f"{SERVING_API_ROUTE_PREFIX}/products/does-not-exist")
hit.status_code, hit.json(), miss.status_code, miss.json()